<a href="https://colab.research.google.com/github/james-kalaw/FlyRank-ML-Intern-Kalaw/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/james-kalaw/FlyRank-ML-Intern-Kalaw/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane: Ranking Signal Analysis (Lane 1).**

I chose this lane because in my Week 1 and Week 2 notebooks I already tested several common SEO beliefs with simple pandas checks (correlation and group-by), and each one either contradicted the common assumption or showed something non-obvious:

- Search volume barely relates to actual traffic (corr = 0.001).
- CTR drops sharply by position tier, yet a raw correlation between `avg_position` and `ctr` is nearly zero (-0.073). The relationship is non-linear, not absent.
- Word count is nearly identical between declining and growing pages.
- Declining pages have the *lowest* median content age, the opposite of the "stale content decays" assumption.

I want to keep testing this way, more systematically, across more of the 44 columns. As a beginner, this lane also builds on tools I already use (`.corr()`, `.groupby()`) while I am still learning what a good research question looks like.

In [1]:
# Load the starter CSV (clones my repo first if the data folder is not there yet)
import os, subprocess
import pandas as pd

DATA = "data/raw/content_refresh_anonymized.csv"
if not os.path.exists(DATA):
    if not os.path.isdir("repo"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/james-kalaw/FlyRank-ML-Intern-Kalaw.git", "repo"],
                       check=True)
    os.chdir("repo")

df = pd.read_csv(DATA)
print(df.shape[0], "rows,", df.shape[1], "columns")

30000 rows, 44 columns


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**What decision does this improve?**
Whether an SEO/content team keeps acting on common assumptions (for example "refresh old content" or "write longer articles"), or redirects effort toward signals that actually show a relationship with performance in this data.

**Who acts on the output, and what do they do?**
A content strategist or SEO lead deciding what to tell their team to prioritize when reviewing pages: lengthen articles, update old pages, or something else, based on what the data supports rather than industry folklore.

**What does a wrong answer cost?**
If my report says a signal matters when it does not (false positive), a team wastes hours acting on it with no payoff, such as refreshing old pages when age turns out not to matter. If I miss a signal that does matter (false negative), the team keeps ignoring something that would have helped. This is a signal report, not an automated tool, so the immediate cost is time and trust, not an automated bad decision. The stakes are real because many pages are affected: roughly half the pages in this sample are trending down, so a team has many candidates and limited time.

**Why does data help at all, instead of a plain rule?**
The rules floating around (longer is better, older content decays, high search volume means high traffic) sound reasonable but are rarely checked. With simple pandas checks I already found that several do not hold in this dataset. A plain rule cannot tell you it is wrong; testing against real data can.

**One-paragraph frame:**
For a content strategist or SEO lead deciding which common assumptions to act on (content age, word count, search volume) versus ignore, we will build a signal report from the anonymized content performance dataset, testing which measured signals show a real relationship with traffic, clicks, and decline, using correlations and grouped comparisons as the method. A wrong call costs wasted team hours chasing signals that do not matter, or continued neglect of signals that do. A plain rule is not enough because several common assumptions have already failed simple checks in this data (for example, content age runs backwards from the "staleness" assumption). We will claim only observed, directional findings in this dataset, never proof of how Google's algorithm works.

**Unit of analysis:** one page (one content item), with its metrics summed over a 90-day window.

**Output:** a signal report, not a trained model. Each tested signal gets a verdict: holds up, doesn't hold up, or only shows up when grouped.

**Why this isn't just "train a model":** a model would tell us which pages look risky, but not which beliefs are wrong. This lane checks the beliefs directly, one signal at a time, so a content lead can stop spending time on the ones that don't hold up.

In [2]:
# How many pages are trending which way (context for the cost of a wrong call)
share = df["trend_direction"].value_counts(normalize=True).mul(100).round(1)
print("Share of pages by trend_direction (%):")
print(share.to_string())

Share of pages by trend_direction (%):
trend_direction
down      54.2
stable    19.9
up        14.6
new        7.5
flat       3.8


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

Three numbers from this dataset already show that common SEO assumptions do not hold up cleanly, which is what Lane 1 investigates systematically:

1. **Content age runs backwards from the "staleness" assumption.** Declining pages have the *youngest* median age (216 days), while stable and growing pages are older (300 and 292 days).
2. **A raw correlation can hide a real effect.** `avg_position` vs `ctr` is only -0.073, yet a grouped comparison (Discovery B in my Week 1 notebook) shows a steep CTR cliff by position tier. The relationship is non-linear and easy to miss with the wrong method.
3. **Search demand and actual traffic are essentially unrelated.** The correlation between `search_volume` and `impressions_90d` is 0.001, so a popular topic does not mean the page itself gets seen.

With 44 columns and only a handful tested so far, there is more folklore left to check against real numbers.

In [3]:
# 2-3 real numbers showing signal analysis is worth pursuing

# 1. A common assumption that appears backwards in this data
age_by_trend = df.groupby("trend_direction")["content_age_days"].median()
print("Median content age (days) by trend direction:")
print(age_by_trend.round(0).to_string())

# 2. A signal that looks absent in a raw correlation
corr_pos_ctr = df["avg_position"].corr(df["ctr"])
print(f"\nRaw correlation, avg_position vs ctr: {corr_pos_ctr:.3f}")

# 3. A widely assumed driver with almost no relationship
corr_sv = df["search_volume"].corr(df["impressions_90d"])
print(f"Correlation, search_volume vs impressions_90d: {corr_sv:.3f}")

Median content age (days) by trend direction:
trend_direction
down      216.0
flat      231.0
new       279.0
stable    300.0
up        292.0

Raw correlation, avg_position vs ctr: -0.073
Correlation, search_volume vs impressions_90d: 0.001


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What my work in this lane WILL be able to say:**
- Observed relationships (or lack of them) between specific measured signals and page performance, *within this anonymized 30,000-row sample*. Example: "in this dataset, declining pages have a lower median content age than stable pages."
- Directional patterns, stated as tendencies, not guarantees. Example: "search volume shows almost no relationship with traffic in this sample," not "search volume never matters."
- Decision-support findings that tell a human where to look first. A signal report is a starting point for investigation, not a final verdict.
- Honest limits of the method. A linear correlation can miss a real but non-linear relationship (as with `avg_position` vs `ctr`), so I will pair correlations with grouped comparisons.

**What my work will NEVER claim:**
- That any signal *causes* a page to decline or grow. Correlations and group comparisons show relationships, not cause and effect. A page could be old and stable for reasons unrelated to its age.
- That any finding describes how Google's ranking algorithm works. The data shows outcomes (clicks, impressions, position), not the algorithm's inner logic.
- That a pattern in these 32 clients' pages applies to all websites, industries, or all of Google Search.
- More certainty than the sample size supports, especially for groups with few pages, where a handful of pages can swing an average.

In [4]:
# Every claim in this notebook is limited to this sample
print(f"Scope of all claims: {df.shape[0]:,} pages from {df['client_id'].nunique()} clients.")

Scope of all claims: 30,000 pages from 32 clients.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.